# Cu FEFF: tune M3GNet, then train a new tunedUniversalXAS head

This notebook does exactly two stages:

1. **Stage A:** fine-tune M3GNet on Cu_FEFF spectra using the existing Cu_FEFF tunedUniversalXAS head only as the supervisor.
2. **Stage B:** freeze that fine-tuned M3GNet, regenerate Cu_FEFF features, then fine-tune a **new** Cu_FEFF model starting from UniversalXAS All_FEFF.

Original checkpoints are never overwritten.


In [1]:
from pathlib import Path
from datetime import datetime
import copy, glob, json, os, random, re, shutil

import dgl
import lightning.pytorch as pl
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pymatgen.core import Lattice, Structure

from matgl import load_model
from matgl.ext.pymatgen import Structure2Graph
try:
    from matgl.graph.compute import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
except Exception:
    from matgl.graph._compute_dgl import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
from matgl.utils.cutoff import polynomial_cutoff
import matgl.layers._basis as mbasis
import matgl.layers._three_body as m3body
import matgl.utils.maths as mmath
from omnixas.model.xasblock import XASBlock

# GPU fixes for MatGL 0.8.x: several helper tensors are otherwise created on CPU.
def _safe_sbf(self, r):
    cutoff = torch.as_tensor(self.cutoff, dtype=r.dtype, device=r.device)
    roots = mbasis.SPHERICAL_BESSEL_ROOTS[: self.max_l, : self.max_n].to(r.device, r.dtype)
    r_c = r.clamp(max=cutoff)
    factor = torch.sqrt(torch.as_tensor(2.0, dtype=r.dtype, device=r.device) / cutoff**3)
    return torch.cat([
        self.funcs[i](r_c[:, None] * roots[i][None, :] / cutoff) * factor / torch.abs(self.funcs[i + 1](roots[i][None, :]))
        for i in range(self.max_l)
    ], axis=1)

def _safe_combine(sbf, shf, max_n, max_l, use_phi):
    if sbf.size(0) == 0:
        return sbf
    if use_phi:
        repeats = torch.repeat_interleave(2 * torch.arange(max_l, device=sbf.device) + 1, max_n)
        blocks = 2 * torch.arange(max_l, device=sbf.device) + 1
    else:
        repeats, blocks = torch.ones(max_l * max_n, dtype=torch.long, device=sbf.device), [1] * max_l
    expanded_sbf = torch.repeat_interleave(sbf, repeats, 1)
    col, idx, start = torch.arange(shf.size(1), device=shf.device), [], 0
    for b in blocks:
        b = int(b.item()) if torch.is_tensor(b) else int(b)
        idx.append(torch.tile(col[start:start + b], [max_n])); start += b
    expanded_shf = torch.index_select(shf, 1, torch.cat(idx))
    return torch.reshape(expanded_sbf * expanded_shf, [-1, max_n * max_l * (max_l if use_phi else 1)])

def _safe_scatter_sum(x, segment_ids, num_segments, dim):
    segment_ids = mmath.broadcast(segment_ids.to(x.device), x, dim)
    size = list(x.size()); size[dim] = 0 if segment_ids.numel() == 0 else num_segments
    return torch.zeros(size, dtype=x.dtype, device=x.device).scatter_add_(dim, segment_ids, x)

mbasis.SphericalBesselFunction._call_sbf = _safe_sbf
mbasis.combine_sbf_shf = _safe_combine
m3body.combine_sbf_shf = _safe_combine
mmath.scatter_sum = _safe_scatter_sum
m3body.scatter_sum = _safe_scatter_sum



In [2]:
SEED = 42
SEEDS = [42]  # keep total runtime small: one M3GNet fine-tune run
pl.seed_everything(SEED, workers=True)
np.random.seed(SEED); random.seed(SEED)
torch.set_float32_matmul_precision("medium")

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "tutorial_omnixas":
    REPO_ROOT = REPO_ROOT.parent

ELEMENT, SPECTRUM_TYPE = "Cu", "FEFF"
DATASET_KEY = f"{ELEMENT}_{SPECTRUM_TYPE}"
ML_DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
ID_SITE_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
RAW_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")) / "materialscloud_omnixas_raw" / "extracted"
M3GNET_MODEL_PATH = REPO_ROOT / "models" / "M3GNet-MP-2021.2.8-PES"
# Stage A supervisor: existing Cu_FEFF tunedUniversalXAS head, not the Cu expert head.
HEAD_MODEL_FAMILY = "tunedUniversalXAS"
HEAD_CKPT_PATH = None  # None = latest best checkpoint under output/training/{HEAD_MODEL_FAMILY}/Cu_FEFF

FEATURE_SCALE = 1000.0
BATCH_SIZE, MAX_EPOCHS, PATIENCE, NUM_WORKERS = 32, 30, 8, 4
MAX_TRAIN_SAMPLES = None  # quick test: 256
MAX_VAL_SAMPLES = None    # quick test: 64
ACCELERATOR = "gpu"

USE_ADAPTER = False  # keep Stage A as pure M3GNet tuning; no adapter by default
TRAIN_BLOCK_INDICES = [1, 2]  # zero-based: blocks 2 and 3
STAGE_SCHEDULE = [
    {"start": 0,  "adapter": 0.0, "block3": 5e-6, "block2": 0.0,  "mlp": 0.0},
    {"start": 10, "adapter": 0.0, "block3": 3e-6, "block2": 1e-6, "mlp": 0.0},
    {"start": 16, "adapter": 0.0, "block3": 1e-6, "block2": 3e-7, "mlp": 0.0},
]
UNFREEZE_MLP_HEAD, MLP_HEAD_TRAIN_MODE = False, "last_linear"
WEIGHT_DECAY, ANCHOR_LAMBDA, DERIV_LAMBDA = 1e-5, 1e-3, 0.05

RUN_ROOT = REPO_ROOT / "output" / "training" / "m3gnetLastBlockFinetuneTunedUniversal" / DATASET_KEY
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("run_root:", RUN_ROOT)


Seed set to 42


run_root: /mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF


In [3]:
def id_sites(split):
    return [(m, int(s)) for m, s in (line.rsplit("_", 1) for line in (ID_SITE_DIR / f"{DATASET_KEY}_{split}.txt").read_text().splitlines() if line.strip())]

def best_head_ckpt():
    if HEAD_CKPT_PATH:
        return Path(HEAD_CKPT_PATH)
    matches = []
    for pat in [
        REPO_ROOT / "output" / "training" / HEAD_MODEL_FAMILY / DATASET_KEY / "runs" / "*" / "best*.ckpt",
        REPO_ROOT / "output" / "training" / HEAD_MODEL_FAMILY / DATASET_KEY / "checkpoints" / "best*.ckpt",
    ]:
        matches += glob.glob(str(pat))
    if not matches:
        raise FileNotFoundError(f"No {HEAD_MODEL_FAMILY} checkpoint found for {DATASET_KEY}")
    return Path(sorted(matches, key=os.path.getmtime)[-1])

def load_head(path):
    state = torch.load(path, map_location="cpu").get("state_dict")
    layers = sorted((int(m.group(1)), tuple(v.shape)) for k, v in state.items() if (m := re.fullmatch(r"model\.(\d+)\.weight", k)) and v.ndim == 2)
    dims = [layers[0][1][1]] + [shape[0] for _, shape in layers]
    head = XASBlock(dims[0], dims[1:-1], dims[-1])
    head.load_state_dict({k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")})
    for p in head.parameters():
        p.requires_grad = False
    return head.eval()

def parse_feff_structure(path):
    abc = angles = None; species, coords = [], []
    site_re = re.compile(r"^\*\s+\d+\s+([A-Z][a-z]?)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)")
    for line in path.read_text(errors="ignore").splitlines():
        if line.startswith("TITLE abc:"): abc = [float(x) for x in line.split(":", 1)[1].split()[:3]]
        elif line.startswith("TITLE angles:"): angles = [float(x) for x in line.split(":", 1)[1].split()[:3]]
        elif (m := site_re.match(line)):
            species.append(m.group(1)); coords.append([float(m.group(2)), float(m.group(3)), float(m.group(4))])
    return Structure(Lattice.from_parameters(*abc, *angles), species, coords, coords_are_cartesian=False)

def load_structure(mid, site):
    if SPECTRUM_TYPE == "VASP":
        return Structure.from_file(RAW_ROOT / "VASP" / ELEMENT / mid / "VASP" / f"{site:03d}_{ELEMENT}" / "POSCAR")
    mdir = RAW_ROOT / "FEFF" / ELEMENT / mid
    return Structure.from_file(mdir / "POSCAR") if (mdir / "POSCAR").exists() else parse_feff_structure(mdir / "FEFF-XANES" / f"{site:03d}_{ELEMENT}" / "feff.inp")

class XASDataset(Dataset):
    def __init__(self, split, max_samples=None):
        self.ids = id_sites(split)
        self.y = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_{split}_y.txt", dtype=np.float32)
        self.anchor = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_{split}_X.txt", dtype=np.float32) / FEATURE_SCALE
        if max_samples:
            self.ids, self.y, self.anchor = self.ids[:max_samples], self.y[:max_samples], self.anchor[:max_samples]
        self.cache = {}
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        mid, site = self.ids[i]; key = (mid, site if SPECTRUM_TYPE == "VASP" else None)
        if key not in self.cache: self.cache[key] = load_structure(mid, site)
        return self.cache[key], site, torch.tensor(self.y[i]), torch.tensor(self.anchor[i])

class GraphBatcher:
    def __init__(self, m3gnet): self.converter = Structure2Graph(m3gnet.element_types, m3gnet.cutoff)
    @staticmethod
    def writable_tensor(array):
        return torch.tensor(np.array(array, copy=True), dtype=torch.float32)
    def graph(self, structure):
        out = self.converter.get_graph(structure)
        g = out[0]
        lat_src = structure.lattice.matrix if len(out) == 2 else (out[1][0] if getattr(out[1], "ndim", 0) == 3 else out[1])
        lat = self.writable_tensor(lat_src)
        g.edata["pbc_offshift"] = g.edata.get("pbc_offshift", g.edata["pbc_offset"].float() @ lat).float()
        frac = self.writable_tensor(structure.frac_coords)
        g.ndata["pos"] = g.ndata.get("pos", g.ndata.get("frac_coords", frac).float() @ lat).float()
        g.edata["bond_vec"], g.edata["bond_dist"] = compute_pair_vector_and_distance(g)
        return g
    def __call__(self, batch):
        graphs, sites, offset = [], [], 0
        for structure, site, _, _ in batch:
            g = self.graph(structure); graphs.append(g); sites.append(offset + site); offset += g.num_nodes()
        return {"graph": dgl.batch(graphs), "site": torch.tensor(sites), "y": torch.stack([b[2] for b in batch]).float(), "anchor": torch.stack([b[3] for b in batch]).float()}


In [4]:
class Adapter(nn.Module):
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 64))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, z): return z + self.net(z)

class M3GNetXAS(nn.Module):
    def __init__(self, m3gnet, head):
        super().__init__(); self.m3gnet, self.head = m3gnet, head
        self.adapter = Adapter() if USE_ADAPTER else nn.Identity()
        self.train_blocks = sorted(set(TRAIN_BLOCK_INDICES))
        for p in self.m3gnet.parameters(): p.requires_grad = False
        for i in self.train_blocks:
            for p in self.m3gnet.three_body_interactions[i].parameters(): p.requires_grad = True
            for p in self.m3gnet.graph_layers[i].parameters(): p.requires_grad = True
        for p in self.head.parameters(): p.requires_grad = False
        self.head_params = []
        if UNFREEZE_MLP_HEAD:
            if MLP_HEAD_TRAIN_MODE != "last_linear": raise ValueError("Only last_linear kept in compact notebook")
            self.head_params = list(next(layer for layer in reversed(self.head) if isinstance(layer, nn.Linear)).parameters())
        self.set_head_trainable(False)
    def set_head_trainable(self, enabled):
        for p in self.head_params: p.requires_grad = enabled
        self.head.eval()
    def train(self, mode=True):
        super().train(mode); self.head.eval(); return self
    def encode(self, g):
        g.edata["rbf"] = self.m3gnet.bond_expansion(g.edata["bond_dist"])
        lg = create_line_graph(g.to("cpu"), self.m3gnet.threebody_cutoff).to(g.device)
        lg.apply_edges(compute_theta_and_phi)
        tb_basis, tb_cutoff = self.m3gnet.basis_expansion(lg), polynomial_cutoff(g.edata["bond_dist"], self.m3gnet.threebody_cutoff)
        with torch.no_grad(): node, edge, state = self.m3gnet.embedding(g.ndata["node_type"], g.edata["rbf"], None)
        for i in range(self.m3gnet.n_blocks):
            ctx = torch.enable_grad() if i in self.train_blocks else torch.no_grad()
            with ctx:
                edge = self.m3gnet.three_body_interactions[i](g, lg, tb_basis, tb_cutoff, node, edge)
                edge, node, state = self.m3gnet.graph_layers[i](g, edge, node, state)
        return node
    def forward(self, g, site):
        z = self.adapter(self.encode(g)[site])
        return self.head(z * FEATURE_SCALE), z

class Lit(pl.LightningModule):
    def __init__(self, model): super().__init__(); self.model = model; self.val_mses = []
    def stage(self):
        s = STAGE_SCHEDULE[0]
        for x in STAGE_SCHEDULE:
            if self.current_epoch >= x["start"]: s = x
        return s
    def cosine_scale(self):
        if MAX_EPOCHS <= 1: return 1.0
        return 0.5 * (1.0 + np.cos(np.pi * min(self.current_epoch, MAX_EPOCHS) / MAX_EPOCHS))
    def scheduled_lr(self, name):
        s = self.stage()
        base = {"adapter": s["adapter"], "block2": s["block3"], "block1": s["block2"], "mlp": s["mlp"]}.get(name, 0.0)
        return 0.0 if base <= 0 else 1e-7 + (base - 1e-7) * self.cosine_scale()
    def on_train_epoch_start(self):
        self.model.set_head_trainable(UNFREEZE_MLP_HEAD and self.stage()["mlp"] > 0)
        for group in self.optimizers().param_groups:
            group["lr"] = self.scheduled_lr(group["name"])
            self.log(f"lr_{group['name']}", group["lr"], on_epoch=True)
    def step(self, batch, split):
        g, site = batch["graph"].to(self.device), batch["site"].to(self.device)
        y, anchor = batch["y"].to(self.device), batch["anchor"].to(self.device)
        pred, z = self.model(g, site)
        mse = ((pred - y) ** 2).mean(); dl = ((torch.diff(pred, dim=1) - torch.diff(y, dim=1)) ** 2).mean(); al = ((z - anchor) ** 2).mean()
        loss = mse + DERIV_LAMBDA * dl + ANCHOR_LAMBDA * al
        self.log(f"{split}_loss", loss, on_epoch=True, prog_bar=True); self.log(f"{split}_mse", mse, on_epoch=True, prog_bar=True)
        if split == "val": self.val_mses.append(((pred - y) ** 2).mean(dim=1).detach())
        return loss
    def training_step(self, b, _): return self.step(b, "train")
    def on_validation_epoch_start(self): self.val_mses = []
    def validation_step(self, b, _): return self.step(b, "val")
    def on_validation_epoch_end(self):
        if self.val_mses: self.log("val_median_mse", torch.cat(self.val_mses).median(), on_epoch=True, prog_bar=True)
    def configure_optimizers(self):
        groups = []
        for i in self.model.train_blocks:
            groups.append({"params": list(self.model.m3gnet.three_body_interactions[i].parameters()) + list(self.model.m3gnet.graph_layers[i].parameters()), "lr": 0.0, "name": f"block{i}"})
        if USE_ADAPTER: groups.append({"params": self.model.adapter.parameters(), "lr": STAGE_SCHEDULE[0]["adapter"], "name": "adapter"})
        if self.model.head_params: groups.append({"params": self.model.head_params, "lr": 0.0, "name": "mlp"})
        return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


In [5]:
head_ckpt = best_head_ckpt(); head = load_head(head_ckpt)
print("head family:", HEAD_MODEL_FAMILY)
print("head checkpoint:", head_ckpt)
base_m3gnet = load_model(str(M3GNET_MODEL_PATH)).model.eval()
train_ds, val_ds = XASDataset("train", MAX_TRAIN_SAMPLES), XASDataset("val", MAX_VAL_SAMPLES)
collate = GraphBatcher(base_m3gnet)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
model = M3GNetXAS(copy.deepcopy(base_m3gnet), copy.deepcopy(head))
print("samples train/val:", len(train_ds), len(val_ds), "trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("probe:", next(iter(val_loader))["y"].shape)


head family: tunedUniversalXAS
head checkpoint: /mnt/c/Users/anton/Desktop/OmniXAS/output/training/tunedUniversalXAS/Cu_FEFF/runs/paper_20260612_213946_735821_seed1090614229_dropout0p0/best-model-epoch=231-val_loss=0.0037.ckpt
samples train/val: 3340 416 trainable: 171154
probe: torch.Size([32, 141])


In [6]:
RUN_RESULTS = []
for seed in SEEDS:
    pl.seed_everything(seed, workers=True); np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    run_dir = RUN_ROOT / f"{datetime.now():%Y%m%d_%H%M%S}_seed{seed}"; run_dir.mkdir(parents=True)
    run_model = M3GNetXAS(copy.deepcopy(base_m3gnet), copy.deepcopy(head))
    ckpt = ModelCheckpoint(run_dir / "checkpoints", filename="best-{epoch:03d}-{val_median_mse:.5f}-{val_mse:.5f}", monitor="val_median_mse", mode="min", save_top_k=1, save_last=True)
    trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator=ACCELERATOR, devices=1, callbacks=[EarlyStopping(monitor="val_median_mse", patience=PATIENCE, mode="min"), ckpt], logger=CSVLogger(str(run_dir), name="logs"), log_every_n_steps=1)
    trainer.fit(Lit(run_model), train_loader, val_loader)
    best_copy = run_dir / "best_finetuned_m3gnet_xas.ckpt"
    if ckpt.best_model_path: shutil.copy2(ckpt.best_model_path, best_copy)
    RUN_RESULTS.append({"seed": seed, "run_dir": str(run_dir), "best_model_path": ckpt.best_model_path, "stable_best_copy": str(best_copy), "best_val_median_mse": float(ckpt.best_model_score.detach().cpu()), "head_model_family": HEAD_MODEL_FAMILY, "head_ckpt_path": str(head_ckpt), "stage_schedule": STAGE_SCHEDULE})

BEST_RUN = min(RUN_RESULTS, key=lambda x: x["best_val_median_mse"])
BEST_TUNED_CKPT, RUN_DIR = BEST_RUN["stable_best_copy"], Path(BEST_RUN["run_dir"])
summary = RUN_ROOT / f"summary_{datetime.now():%Y%m%d_%H%M%S}.json"; summary.write_text(json.dumps({"best": BEST_RUN, "all_runs": RUN_RESULTS}, indent=2))
print("BEST_RUN:", json.dumps(BEST_RUN, indent=2))


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type      | Params | Mode  | FLOPs
----------------------------------------------------
0 | model | M3GNetXAS | 927 K  | train | 0    
----------------------------------------------------
171 K     Trainable params
756 K     Non-trainable params
927 K     Total params
3.710     Total estimated model params size (MB)
2         Modules in train mode
185       Modules in eval mode
0         Total Flops


Sanity Checking: |                                                                                            …

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 185 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |                                                                                                   …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

BEST_RUN: {
  "seed": 42,
  "run_dir": "/mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42",
  "best_model_path": "/mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/checkpoints/best-epoch=002-val_median_mse=0.00220-val_mse=0.00372.ckpt",
  "stable_best_copy": "/mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/best_finetuned_m3gnet_xas.ckpt",
  "best_val_median_mse": 0.0022029776591807604,
  "head_model_family": "tunedUniversalXAS",
  "head_ckpt_path": "/mnt/c/Users/anton/Desktop/OmniXAS/output/training/tunedUniversalXAS/Cu_FEFF/runs/paper_20260612_213946_735821_seed1090614229_dropout0p0/best-model-epoch=231-val_loss=0.0037.ckpt",
  "stage_schedule": [
    {
      "start": 0,
      "adapter": 0.0,
      "block3": 5e-06,
      "block2": 0.0,
      "mlp": 0.0
    },
    {
      "star

In [7]:
@torch.no_grad()
def predict_cached(split):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    x = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_{split}_X.txt", dtype=np.float32)
    y = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_{split}_y.txt", dtype=np.float32)
    h = copy.deepcopy(head).to(device).eval()
    preds = []
    for i in range(0, len(x), BATCH_SIZE * 8):
        xb = torch.tensor(x[i:i + BATCH_SIZE * 8], dtype=torch.float32, device=device)
        preds.append(h(xb).cpu().numpy())
    return np.concatenate(preds), y


def predict_head_array(model, X, batch_size=1024):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i + batch_size], dtype=torch.float32, device=device)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds)


def eta(pred, target, train_y):
    base = np.repeat(train_y.mean(axis=0, keepdims=True), len(target), axis=0)
    med = float(np.median(np.mean((target - pred) ** 2, axis=1)))
    bmed = float(np.median(np.mean((target - base) ** 2, axis=1)))
    return {"eta": bmed / med, "median_mse": med, "baseline_median_mse": bmed}


def load_tuned(path):
    m = M3GNetXAS(copy.deepcopy(base_m3gnet), copy.deepcopy(head))
    state = torch.load(path, map_location="cpu").get("state_dict")
    m.load_state_dict({k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")}, strict=True)
    return m


train_y = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_train_y.txt", dtype=np.float32)


## Stage B: train new Cu_FEFF ExpertXAS heads on tuned M3GNet features

This tests whether the fine-tuned M3GNet representation becomes useful when the downstream Cu model is trained directly on the new feature space.

Unlike tunedUniversalXAS, ExpertXAS starts from scratch with the Cu_FEFF expert architecture.


In [8]:
EXPORT_AFTER_ADAPTER = False  # Stage B uses raw fine-tuned M3GNet features.

@torch.no_grad()
def tuned_features(split, eval_model):
    ds = XASDataset(split)
    loader = DataLoader(ds, BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    eval_model = eval_model.to(device).eval()
    old_blocks, eval_model.train_blocks = list(eval_model.train_blocks), []
    feats, targets = [], []
    for b in loader:
        g, site = b["graph"].to(device), b["site"].to(device)
        z = eval_model.encode(g)[site]
        if EXPORT_AFTER_ADAPTER:
            z = eval_model.adapter(z)
        feats.append((z * FEATURE_SCALE).cpu().numpy())
        targets.append(b["y"].numpy())
    eval_model.train_blocks = old_blocks
    return np.concatenate(feats), np.concatenate(targets)


tuned_encoder = load_tuned(BEST_TUNED_CKPT)
X_tuned, y_tuned = {}, {}
for split in ["train", "val", "test"]:
    X_tuned[split], y_tuned[split] = tuned_features(split, tuned_encoder)
    print(split, X_tuned[split].shape, y_tuned[split].shape)

feature_dir = RUN_DIR / "tuned_encoder_features"
feature_dir.mkdir(exist_ok=True)
for split in ["train", "val", "test"]:
    np.savetxt(feature_dir / f"{DATASET_KEY}_{split}_X_tuned_encoder.txt", X_tuned[split])
    np.savetxt(feature_dir / f"{DATASET_KEY}_{split}_y.txt", y_tuned[split])
print("saved tuned features to", feature_dir)


train (3340, 64) (3340, 141)
val (416, 64) (416, 141)
test (416, 64) (416, 141)
saved tuned features to /mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/tuned_encoder_features


In [9]:
from omnixas.data import MLData, MLSplits
from omnixas.model.xasblock_regressor import XASBlockRegressor

RUN_EXPERT_RETRAIN_ON_TUNED_FEATURES = True
EXPERT_DIMS = [600, 600, 400]  # Cu_FEFF expert architecture from train_paper_models.py
# Four ExpertXAS runs + one M3GNet run above = five total training runs.
EXPERT_RETRAIN_CONFIGS = [
    {"seed": 46, "dropout": 0.50, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 47, "dropout": 0.40, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 48, "dropout": 0.30, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 49, "dropout": 0.25, "lr": 7e-4, "scheduler": "cosine"},
    {"seed": 50, "dropout": 0.10, "lr": 7e-4, "scheduler": "cosine"},
    # Extra focused runs around the current best region.
    {"seed": 51, "dropout": 0.50, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 52, "dropout": 0.45, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 53, "dropout": 0.40, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 54, "dropout": 0.35, "lr": 1e-3, "scheduler": "cosine"},
    {"seed": 55, "dropout": 0.30, "lr": 7e-4, "scheduler": "cosine"},
]
EXPERT_RETRAIN_MAX_EPOCHS = 800
EXPERT_RETRAIN_PATIENCE = 60
EXPERT_RETRAIN_BATCH_SIZE = 32

splits_tuned = MLSplits(
    train=MLData(X=X_tuned["train"], y=y_tuned["train"]),
    val=MLData(X=X_tuned["val"], y=y_tuned["val"]),
    test=MLData(X=X_tuned["test"], y=y_tuned["test"]),
)


def fit_expert(seed, dropout, lr, scheduler):
    pl.seed_everything(seed, workers=True)
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    XASBlock.DROPOUT = dropout
    out_dir = RUN_DIR / "expert_retrain" / f"seed{seed}_dropout{str(dropout).replace('.', 'p')}_lr{lr:g}_{scheduler}"
    reg = XASBlockRegressor(
        directory=str(out_dir),
        input_dim=64,
        hidden_dims=EXPERT_DIMS,
        output_dim=y_tuned["train"].shape[1],
        initial_lr=lr,
        batch_size=EXPERT_RETRAIN_BATCH_SIZE,
        max_epochs=EXPERT_RETRAIN_MAX_EPOCHS,
        early_stopping_patience=EXPERT_RETRAIN_PATIENCE,
        use_lr_finder=False,
        monitor_metric="val_median_mse",
        shuffle=True,
        lr_scheduler=scheduler,
        cosine_t_max=EXPERT_RETRAIN_MAX_EPOCHS,
        cosine_eta_min=1e-6,
        warmup_epochs=10,
        warmup_start_factor=0.1,
        onecycle_max_lr=lr,
    )
    reg.fit(splits_tuned).load("best")
    val = eta(predict_head_array(reg.model.model, X_tuned["val"]), y_tuned["val"], y_tuned["train"])
    test = eta(predict_head_array(reg.model.model, X_tuned["test"]), y_tuned["test"], y_tuned["train"])
    return {
        "seed": seed, "dropout": dropout, "lr": lr, "scheduler": scheduler, "run_dir": str(out_dir),
        "best_model_path": reg.cfg.fetch_checkpoint("best"),
        "val_eta": val["eta"], "val_median_mse": val["median_mse"],
        "test_eta": test["eta"], "test_median_mse": test["median_mse"],
    }


EXPERT_RETRAIN_RESULTS = []
if RUN_EXPERT_RETRAIN_ON_TUNED_FEATURES:
    for cfg in EXPERT_RETRAIN_CONFIGS:
        result = fit_expert(**cfg)
        EXPERT_RETRAIN_RESULTS.append(result)
        print(result)
    (RUN_DIR / "expert_retrain_results.json").write_text(json.dumps(EXPERT_RETRAIN_RESULTS, indent=2))


Seed set to 46
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/expert_retrain/seed46_dropout0p5_lr0.001_cosine exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 699 K  | train | 0    
---------------------------------------------------
699 K     Trainable params
0         Non-trainable params


Sanity Checking: |                                                                                            …

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                                   …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

2026-06-30 17:00:02.125 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/expert_retrain/seed55_dropout0p3_lr0.0007_cosine/best-model-epoch=283-val_median_mse=0.0022.ckpt
2026-06-30 17:00:02.126 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.0034499550238251686
2026-06-30 17:00:02.155 | INFO     | omnixas.model.xasblock_regressor:load:265 - Loading model from /mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/expert_retrain/seed55_dropout0p3_lr0.0007_cosine/best-model-epoch=283-val_median_mse=0.0022.ckpt


{'seed': 55, 'dropout': 0.3, 'lr': 0.0007, 'scheduler': 'cosine', 'run_dir': '/mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/expert_retrain/seed55_dropout0p3_lr0.0007_cosine', 'best_model_path': '/mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_155922_seed42/expert_retrain/seed55_dropout0p3_lr0.0007_cosine/best-model-epoch=283-val_median_mse=0.0022.ckpt', 'val_eta': 5.6571900887150885, 'val_median_mse': 0.002206187229603529, 'test_eta': 5.317858242343939, 'test_median_mse': 0.002403510268777609}


In [10]:
import pandas as pd


def best_ckpt_for_family(family):
    matches = []
    for pat in [
        REPO_ROOT / "output" / "training" / family / DATASET_KEY / "runs" / "*" / "best*.ckpt",
        REPO_ROOT / "output" / "training" / family / DATASET_KEY / "checkpoints" / "best*.ckpt",
    ]:
        matches += glob.glob(str(pat))
    if not matches:
        raise FileNotFoundError(f"No {family} checkpoint found for {DATASET_KEY}")
    return Path(sorted(matches, key=os.path.getmtime)[-1])


def predict_cached_with_head(head_model, split):
    x = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_{split}_X.txt", dtype=np.float32)
    y = np.loadtxt(ML_DATA_DIR / f"{DATASET_KEY}_{split}_y.txt", dtype=np.float32)
    return predict_head_array(copy.deepcopy(head_model), x), y


def load_expert_from_ckpt(path):
    model = XASBlock(64, EXPERT_DIMS, y_tuned["train"].shape[1])
    state = torch.load(path, map_location="cpu")["state_dict"]
    model.load_state_dict({k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")})
    return model


def score_expert_ckpt(path):
    model = load_expert_from_ckpt(path)
    val = eta(predict_head_array(model, X_tuned["val"]), y_tuned["val"], y_tuned["train"])
    test = eta(predict_head_array(model, X_tuned["test"]), y_tuned["test"], y_tuned["train"])
    return {
        "path": str(path),
        "val_eta": val["eta"],
        "val_median_mse": val["median_mse"],
        "test_eta": test["eta"],
        "test_median_mse": test["median_mse"],
    }


# Consider every new ExpertXAS checkpoint already trained under this experiment root, not just this cell's latest runs.
expert_ckpts = sorted(set(RUN_ROOT.glob("*/expert_retrain/*/best*.ckpt")))
if not expert_ckpts:
    raise FileNotFoundError(f"No ExpertXAS retrain checkpoints found under {RUN_ROOT}")
EXPERT_CANDIDATES = [score_expert_ckpt(path) for path in expert_ckpts]
best_expert_run = max(EXPERT_CANDIDATES, key=lambda x: x["val_eta"])
(RUN_ROOT / "all_expert_retrain_candidates_scored_on_current_features.json").write_text(json.dumps(EXPERT_CANDIDATES, indent=2))
print("best new ExpertXAS by val eta:")
print(json.dumps(best_expert_run, indent=2))

old_expert_head = load_head(best_ckpt_for_family("expertXAS"))
best_expert = load_expert_from_ckpt(best_expert_run["path"])

rows = []
for split in ["val", "test"]:
    pred, y = predict_cached_with_head(old_expert_head, split)
    rows.append({"split": split, "model": "old_M3GNet + frozen_ExpertXAS", **eta(pred, y, train_y)})

    pred = predict_head_array(best_expert, X_tuned[split])
    rows.append({"split": split, "model": "tuned_M3GNet + best_new_ExpertXAS", **eta(pred, y_tuned[split], y_tuned["train"])})

final_df = pd.DataFrame(rows)
display(final_df)
final_df.to_csv(RUN_DIR / "final_eta_comparison_expert_retrain.csv", index=False)


best new ExpertXAS by val eta:
{
  "path": "/mnt/c/Users/anton/Desktop/OmniXAS/output/training/m3gnetLastBlockFinetuneTunedUniversal/Cu_FEFF/20260630_130608_seed42/expert_retrain/seed43_dropout0p25_lr0.001/best-model-epoch=363-val_median_mse=0.0021.ckpt",
  "val_eta": 6.070169744836365,
  "val_median_mse": 0.002056090859696269,
  "test_eta": 5.965404370191625,
  "test_median_mse": 0.002142608631402254
}


,split,model,eta,median_mse,baseline_median_mse
0,val,old_M3GNet + frozen_ExpertXAS,5.345876,0.002335,0.012481
1,val,tuned_M3GNet + best_new_ExpertXAS,6.070170,0.002056,0.012481
2,test,old_M3GNet + frozen_ExpertXAS,4.882931,0.002618,0.012782
3,test,tuned_M3GNet + best_new_ExpertXAS,5.965404,0.002143,0.012782
